# Temporal-v3: графики для диплома

Это **только оформление уже завершённой независимой оценки**: нет обучения, inference, генерации данных или настройки параметров. GPU не нужен; в Colab выбери CPU.

Запускай ячейки сверху вниз (здесь можно «Выполнить все»). Нужно только `numpy` и `matplotlib`, обычно уже установленные в Colab.

Источник: итоговый `summary/payload.json`, 1998 новых каскадов одного Facebook-графа, frozen seeds 7026/7027/7028. Это 1998, **не 5994 независимых примера**.

На Colab: сначала смонтируй Drive отдельной ячейкой, если он ещё не смонтирован:

```python
from google.colab import drive
drive.mount('/content/drive')
```

Notebook найдёт итоговый JSON на Drive. Локально он использует копию в репозитории. Можно задать `SOURCE_PATH` и `OUTPUT_DIR` в отдельной ячейке перед запуском. Рисунки сохраняются в PNG (300 dpi) и SVG; одноимённые **рисунки** при повторном запуске заменяются, исходные результаты не меняются.

Все материалы в `thesis_report/`: текст и формулы — `text/`, схемы — `diagrams/`, исходный JSON — `data/`, экспортированные рисунки — `figures/`.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import matplotlib.pyplot as plt

EXPECTED_SHA256 = '980e20c405503cde35ff8f89db934989c994390fd3d203db73b15ac303328030'
DRIVE_SUMMARY = Path('/content/drive/MyDrive/diffusion-sources/reports/runs/temporal_v3_independent_evaluation/v1/summary/payload.json')
LOCAL_RELATIVE = Path('thesis_report/data/independent_summary.json')
if 'SOURCE_PATH' in globals():
    SUMMARY_PATH = Path(SOURCE_PATH)
else:
    options = [DRIVE_SUMMARY, *[p / LOCAL_RELATIVE for p in [Path.cwd(), *Path.cwd().parents]],
               Path('/content/diffusion-sources') / LOCAL_RELATIVE]
    SUMMARY_PATH = next((p for p in options if p.is_file()), None)
    if SUMMARY_PATH is None:
        raise FileNotFoundError('Не найден итоговый JSON. Смонтируй Drive или задай SOURCE_PATH = Path(путь).')
raw = SUMMARY_PATH.read_bytes()
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
    raise ValueError('SHA-256 отличается от утверждённого summary. Не подменяй результат другим запуском.')
summary = json.loads(raw)

def validate_summary(data):
    if (data['evaluation_role'] != 'independent_confirmation' or data['exploratory'] is not False
        or data['bootstrap_n'] != 1998 or data['seeds'] != [7026, 7027, 7028]
        or data['beta'] != 0.5 or data['count_unchanged'] is not True):
        raise ValueError('Несовместимый протокол независимой оценки.')
    fields = ['snapshot_f1', 'early_expected_f1', 'temporal_f1']
    for group_key in ['by_k', 'by_candidates']:
        groups = data[group_key]
        if sum(g['n_per_seed'] for g in groups.values()) != 1998:
            raise ValueError('Размеры групп не дают 1998 каскадов.')
        for field in fields:
            weighted = sum(g[field] * g['n_per_seed'] for g in groups.values()) / 1998
            if not np.isclose(weighted, data['means'][field], atol=1e-12, rtol=0):
                raise ValueError('Средние по группам не согласованы с общим средним.')
    for field in fields:
        values = [data['per_seed'][str(s)]['means'][field] for s in data['seeds']]
        if (not np.isfinite(values).all() or not all(0 <= v <= 1 for v in values)
            or not np.isclose(np.mean(values), data['means'][field], atol=1e-12, rtol=0)
            or not np.isclose(np.std(values, ddof=1), data['sample_sd'][field], atol=1e-12, rtol=0)):
            raise ValueError('Некорректные значения F1 или SD.')
    for field, control, stage in [('temporal_minus_snapshot', 'snapshot_f1', 'primary'),
                                  ('temporal_minus_early', 'early_expected_f1', 'secondary')]:
        delta = data['means']['temporal_f1'] - data['means'][control]
        lo, hi = data[stage]['ci']
        if (not np.isclose(delta, data['means'][field], atol=1e-12, rtol=0)
            or not np.isclose(delta, data[stage]['delta_f1'], atol=1e-12, rtol=0)
            or not np.isfinite([lo, hi]).all() or not lo <= delta <= hi):
            raise ValueError('Несогласованные разницы F1 или доверительные интервалы.')

validate_summary(summary)
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = (Path('/content/drive/MyDrive/diffusion-sources/thesis_report/figures')
                  if SUMMARY_PATH == DRIVE_SUMMARY else SUMMARY_PATH.resolve().parent.parent / 'figures')
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Источник:', SUMMARY_PATH)
print('SHA-256 проверен; 1998 каскадов, 3 frozen checkpoints; параметры не меняем.')
print('Папка рисунков:', OUTPUT_DIR)

## Оформление

Цвета методов одинаковы на всех рисунках. Для долей ось начинается с нуля. SD — разброс по трём seed; это **не** 95%-й доверительный интервал. Интервалы разниц F1 берём из сохранённого парного bootstrap, не пересчитываем из трёх seed.

In [ ]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'svg.fonttype': 'none', 'figure.facecolor': 'white'})
METHODS = [('snapshot_f1', 'Snapshot', '#56738C'),
           ('early_expected_f1', 'Early-only (expected)', '#D58B35'),
           ('temporal_f1', 'Temporal', '#25866E')]
FIGURES = {}

def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / (name + '.png'), dpi=300, bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / (name + '.svg'), bbox_inches='tight')
    FIGURES[name] = fig
    if plt.get_backend().lower() != 'agg':
        plt.show()
    plt.close(fig)
    print('Сохранено:', name + '.png / .svg')

def grouped_f1(groups, keys, xlabel, title):
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(keys))
    width = 0.24
    for j, (field, label, color) in enumerate(METHODS):
        heights = [groups[key][field] for key in keys]
        bars = ax.bar(x + (j - 1) * width, heights, width, label=label, color=color)
        ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_xticks(x, [str(key) + '\n(n=' + str(groups[key]['n_per_seed']) + ')' for key in keys])
    ax.set(xlabel=xlabel, ylabel='Средний F1 (больше — лучше)', ylim=(0, 0.8), title=title)
    ax.legend(loc='upper left', ncol=3, fontsize=9)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    return fig

## Рисунок 1. Общее качество

**F1** — баланс точности и полноты поиска источников, от 0 до 1; больше — лучше. Столбцы — средние по трём frozen seed; чёрные точки — отдельные seed. Усы — ±1 sample SD, не CI.

Snapshot: 0,3442; Early-only: 0,4349; Temporal: 0,5237. Temporal улучшает поиск, но F1 не равен доле полностью правильных наборов.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
values = [summary['means'][field] for field, _, _ in METHODS]
sd = [summary['sample_sd'][field] for field, _, _ in METHODS]
bars = ax.bar(range(3), values, color=[c for _, _, c in METHODS],
              yerr=sd, capsize=5, error_kw={'elinewidth': 1.3})
ax.bar_label(bars, fmt='%.4f', padding=9)
for j, (field, _, _) in enumerate(METHODS):
    points = [summary['per_seed'][str(s)]['means'][field] for s in summary['seeds']]
    ax.scatter(j + np.array([-0.06, 0, 0.06]), points, color='black', s=16, zorder=4)
ax.set_xticks(range(3), [label for _, label, _ in METHODS])
ax.set(ylabel='Средний F1 (больше — лучше)', ylim=(0, 0.65),
       title='Независимая оценка: 1998 новых каскадов')
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
save_figure(fig, '01_overall_f1')

## Рисунок 2. Прирост и неопределённость

**ΔF1** — F1 Temporal минус F1 контроля. Точка — средняя разница, отрезок — сохранённый 95%-й парный bootstrap CI. Нулевая линия — отсутствие прироста.

Оба интервала полностью правее нуля: Temporal−Snapshot +0,1796; Temporal−Early-only +0,0889. CI относится к новым каскадам при фиксированных графе, протоколе и трёх checkpoint, не к любым новым сетям.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for y, stage in enumerate(['primary', 'secondary']):
    result = summary[stage]
    value = result['delta_f1']
    lo, hi = result['ci']
    ax.errorbar(value, y, xerr=np.array([[value - lo], [hi - value]]),
                fmt='o', color='#25866E', capsize=6, markersize=8)
    ax.annotate(f'{value:+.4f}  [ {lo:.4f}; {hi:.4f} ]', (value, y),
                xytext=(0, 16), textcoords='offset points', ha='center', fontsize=10)
ax.axvline(0, color='gray', linestyle='--')
ax.set_yticks([0, 1], ['Temporal − Snapshot', 'Temporal − Early-only'])
ax.set(xlabel='ΔF1 и 95% парный bootstrap CI', xlim=(-0.015, 0.23),
       ylim=(-0.6, 1.7), title='Прирост на независимых каскадах')
ax.grid(axis='x', alpha=0.2)
save_figure(fig, '02_paired_deltas')

## Рисунок 3. Число источников

**k** — истинное число источников, не предсказанное. В каждой группе 666 каскадов. Результаты усреднены по трём seed.

Temporal: 0,4070 / 0,5450 / 0,6192 для k=1/2/3. Добавка сверх Early-only наиболее велика при k=3 (+0,1455). Сравнение групп описательное: отдельных тестов значимости для групп не проводили.

In [ ]:
fig = grouped_f1(summary['by_k'], ['1', '2', '3'],
                 'Истинное число источников k', 'Качество по числу источников')
save_figure(fig, '03_f1_by_k')

## Рисунок 4. Размер множества кандидатов

**Candidate set** — вершины, среди которых выбираем источники. n — число каскадов в группе, одинаковое для каждого seed. Группа «1–10» в этом протоколе фактически содержит 5–10 кандидатов, поскольку минимальный размер — 5.

С ростом множества кандидатов качество снижается у всех методов; Temporal сохраняет преимущество во всех показанных группах. При 51+ кандидатах F1 Temporal = 0,3588 — это оставшаяся трудная область, а не основание исключать такие примеры.

In [ ]:
fig = grouped_f1(summary['by_candidates'], ['1-10', '11-20', '21-50', '51+'],
                 'Число кандидатов', 'Качество по размеру множества кандидатов')
save_figure(fig, '04_f1_by_candidates')

## Рисунок 5. Устойчивость по seed

**Seed** здесь обозначает случайность обучения frozen модели. Все три модели оцениваются на одних и тех же 1998 каскадах. Это не три независимых датасета.

Temporal даёт F1 0,5218–0,5248 и превосходит оба контроля для каждого checkpoint. Три seed — ограниченная проверка устойчивости, не гарантия для всех будущих обучений.

In [ ]:
groups = {str(s): {**summary['per_seed'][str(s)]['means'], 'n_per_seed': 1998}
          for s in summary['seeds']}
fig = grouped_f1(groups, [str(s) for s in summary['seeds']],
                 'Seed обучения frozen checkpoint', 'Качество каждого frozen checkpoint')
save_figure(fig, '05_f1_by_seed')

## Рисунок 6. Точность и близость локализации

Слева **Exact set** — доля полностью совпавших наборов; **Hit@1/2-hop** — средняя доля истинных источников, для которых есть предсказание не далее одного/двух рёбер (точное попадание тоже учитывается); **Count accuracy** — доля примеров с верным количеством источников. Это разные показатели.

Справа — среднее симметричное расстояние между наборами, в рёбрах; меньше — лучше. Это расстояния до ближайшей вершины, не взаимно-однозначное сопоставление.

Exact set: 6,96% → 22,79%; Hit@1-hop: 51,47% → 82,10%; расстояние: 1,1567 → 0,6369. Count accuracy не изменяется (70,69%), поскольку temporal-поправка меняет ранжирование, а не количество. Early-only здесь не рисуем: сохранены expected F1, а не отдельные наборы предсказаний для геометрии.

In [ ]:
fig, (ax, distance_ax) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [2.3, 1]})
metric_keys = ['exact_set_accuracy', 'hit_at_1_hop', 'hit_at_2_hop', 'count_accuracy']
x = np.arange(len(metric_keys))
for j, (method, label, color) in enumerate([('snapshot', 'Snapshot', '#56738C'), ('temporal', 'Temporal', '#25866E')]):
    means = summary['metrics'][method]['mean']
    bars = ax.bar(x + (j - 0.5) * 0.35, [100 * means[k] for k in metric_keys],
                  width=0.35, label=label, color=color)
    ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
    dbars = distance_ax.bar(j, means['symmetric_set_distance'], color=color)
    distance_ax.bar_label(dbars, fmt='%.3f', padding=3)
ax.set_xticks(x, ['Exact set', 'Hit@1-hop', 'Hit@2-hop', 'Count accuracy'])
ax.set(ylabel='Среднее значение, % (больше — лучше)', ylim=(0, 112), title='Точность и покрытие источников')
ax.legend(loc='upper left')
distance_ax.set_xticks([0, 1], ['Snapshot', 'Temporal'])
distance_ax.set(ylabel='Расстояние, рёбра (меньше — лучше)', ylim=(0, 1.45), title='Симметричное расстояние')
for axis in (ax, distance_ax):
    axis.grid(axis='y', alpha=0.2)
    axis.set_axisbelow(True)
save_figure(fig, '06_localization')
print('Готово: 6 графиков в PNG и SVG. Исходные результаты не изменены.')